# Idea:

Building an optimizer that takes into account the hessian information instead of making it on time (like cosine annealing etc) and changes the LR based upon the curvature, and we estimate the hessian using the following:

$$
∇L(x + \Delta) ≈ ∇L(x) + ϵ H(\Delta)
$$

$Δ \to 0$

the above is the taylor series linearization,

$$
H(x) = \frac{∇L(x + \Delta) - ∇L(x)} \Delta
$$

But remember that hessian is actually a matrix, we cant find any curvature information of an area using a matrix hence to actually "understand" the curvature, we use the rayleigh quotient to actually quantify the curavture as it is nothing but the eigenvalue and is not affected by the twisting of the vector space due to the linear transformation occuring due to the hessian, a higher eigenvalue translates to a higher rate of change of slope, and the sign of the eigenvalue dictate the maxima/minima/plane

- +ve eigenvalue $\to$ MINIMUM
- -ve eigenvalue $\to$ MAXIMUM
- close to 0 eigenvalue $\to$ PLANE

$$
κ = \frac{\nabla L^T(x) H(x) \nabla L(x)}{\nabla L^T(x) \nabla L(x)}
$$

But now, we dont have any data of the direction in which we need to flow, so let
$$
\Delta = ϵ∇L(x)
$$
where $ϵ \to 0$

This is equivalent to nesterov momentum "foresight" (check Part 3: CNN Puremath)

$$
H(x)\nabla L(x) = \frac{∇L(x + ϵ∇L(x)) - ∇L(x)} ϵ
$$

hence overall, the expression becomes:

$$
κ = \frac{\nabla L^T(x) \{\frac{∇L(x + ϵ∇L(x)) - ∇L(x)} ϵ\}}{\nabla L^T(x) \nabla L(x)}
$$

let $\nabla L(x) = g$

$$
κ = \frac{g^T \{\frac{∇L(x + ϵg) - g} ϵ\}}{g^T g} \\
κ = \frac{g^T∇L(x + ϵg) - ||g||^2}{||g||^2}
$$

Note that we capture both curvature and gradient data ($||g||$) and both these parameters have a say on the final LR

Now, suppose we are working with batched data meaning our g values are not the actual, true grads of the entire dataset but only a batch/sample of the data, for which the loss landscape might be different than the actual dataset why? because in a particular batch the number of items in each label of the data might be at different concentrations which affect the updates of the learnable params,

for example we have 1000 images of a husky in our data and 1000 images of a golden retreiver but in one batch we have 20 images of husky and 70 images of the other class, and this uneven distribution of labels directly reflect in the parameters the model wants to update, which might be different that the true gradients of the actual complete dataset.

$g_{true}$ = gradients of the entire datset,

Hence to overcome this problem we implement an EWMA (check out Adam section in Part 2: Logistic Regression)

$$
\bar κ_{t} = \beta \bar κ_{t-1} + (1-\beta)κ_t
$$

Now, we need to create a function of the LR using this quantity $\bar \kappa_{t}$ but how?

what we want to do is:


$$
\begin{array}{|l|c|r|}
\hline
\text{κ} & \text{||g||} & \text{Optmal η value change} \\ \hline
\text{+ve High}       & \text{High}               &\text{decrease}                 \\ \hline
\text{-ve High}       & \text{Low}               & \text{increase}               \\ \hline
\text{+ve Low}       & \text{High}               & \text{increase}               \\ \hline
\text{-ve Low}       & \text{Low}               & \text{increase}                \\ \hline
\text{-ve High}       & \text{High}               &\text{decrease}                 \\ \hline
\text{+ve High}       & \text{Low}               & \text{decrease}                \\ \hline
\text{-ve Low}       & \text{High}               & \text{increase}                \\ \hline
\text{+ve Low}       & \text{Low}               & \text{increase}                \\ \hline
\end{array}
$$

TODO: Now we want to make a function that captures this behaviour

Define a function:
$$
risk(\bar \kappa, ||g||)
$$












Right now, our optimizer is only looking at a certian direction: gradient $g$, we want to look at all the other directions in the subspace so instead of using rayleigh quotient we can use Lanczos iteration using krylov subspace and amortization for easier computing but my linear algebra knowledge is not on par with this idea, hence I will program this idea later, but right now, I'll use the rayleigh quotient and define the learning rate scheduler using a simple function inspired from Adam, to reiterate, this form has no inherent mathematical meaning to it, its still a work under progress and this form of the optimizer is no better than me guessing, I will work on the optimizer after finishing the CNN puremath, part 3 of the puremath series.

$$
||\bar g_{t}|| = \alpha ||\bar g_{t-1}|| + (1-\alpha)||g_t||
$$

$$
f(\bar \kappa _t, ||\bar g_t||) = clamp(\frac{\eta_0||\bar g_t||}{ln({1 + e^{\bar \kappa_t}})}, \eta_{min}, \eta_{max})
$$

we are using softplus and not squareroot because $\bar κ_t$ maybe -ve and greater than 1 in magnitude, hence the square root may be negative which isnt in the domain of the function, so we use softplus activation function

In [ ]:
import numpy as np
class CurvatureAndGradientBasedOptimizer:
  def __init__(self, function_to_optimize, base_learning_rate : float, minimum_learning_rate : float, maximum_learning_rate : float, alpha : float, beta : float):
    self.base_lr = base_learning_rate
    self.min_lr = minimum_learning_rate
    self.max_lr = maximum_learning_rate
    self.fn = function_to_optimize
    self.alpha = alpha
    self.beta = beta
    self.t = 0
    self.x = None
    self.g = None
    self.g_bar = 0.0
    self.kappa_bar = 0.0
  def calculate_gradients(self):
    return self.fn.gradients(self.x)
  def calculate_hessian_change(self, epsilon=1e-4):
    self.epsilon = epsilon
    return (self.fn.gradients(self.x + self.epsilon * self.g) - self.g)/self.epsilon
  def calculate_curvature(self):
    return self.g.T @ self.calculate_hessian_change() / (self.g.T @ self.g)
  def track_curvature_and_gradient(self):
    self.t += 1
    self.g_bar = self.alpha * self.g_bar + (1 - self.alpha) * self.g
    kappa = self.calculate_curvature()
    self.kappa_bar = self.beta * self.kappa_bar + (1 - self.beta) * kappa
    kappa_hat = self.kappa_bar / (1 - self.beta ** self.t)
    g_hat = self.g_bar / (1 - self.alpha ** self.t)
    return kappa_hat, g_hat
  def get_learning_rate(self):
    return float(np.clip(self.base_lr * np.linalg.norm(self.g_bar) / max(np.log1p(np.exp(self.kappa_bar)), 1e-8), self.min_lr, self.max_lr))
  def true_hessian(self, x, epsilon=1e-5):
    n = len(x)
    H = np.zeros((n, n))
    for i in range(n):
        e_i       = np.zeros(n)
        e_i[i]    = 1.0
        g_plus    = self.fn.gradients(x + epsilon * e_i)
        g_minus   = self.fn.gradients(x - epsilon * e_i)
        H[:, i]   = (g_plus - g_minus) / (2 * epsilon)
    return H
  def verify_curvature(self, x):
    g = self.fn.gradients(x)
    H = self.true_hessian(x)

    # true Rayleigh quotient
    kappa_true = (g @ H @ g) / (g @ g)

    # your FD estimate
    self.x = x
    self.g = g
    kappa_fd = self.calculate_curvature()

    print(f"True κ:  {kappa_true:.6f}")
    print(f"FD κ:    {kappa_fd:.6f}")
    print(f"Error:   {abs(kappa_true - kappa_fd):.2e}")
  def step(self, x):
    self.x = x
    self.g = self.calculate_gradients()
    kappa_hat, g_hat = self.track_curvature_and_gradient()
    eta = self.get_learning_rate()
    x_new = self.x - eta * self.g
    return x_new, eta, kappa_hat, g_hat


In [ ]:
class Quadratic:
  def function(self, x : list):
    return x[0]**2 + 10 * x[1]**2
  def gradients(self, x):
    return np.array([2*x[0], 20*x[1]])

In [ ]:
optimizer = CurvatureAndGradientBasedOptimizer(Quadratic(), 0.1, 0.001, 0.1, 0.9, 0.9)

x = np.random.rand(2)

for i in range(100):
  x, eta, kappa, g = optimizer.step(x)
  print(f"At location: ({x[0]:.2f}, {x[1]:.2f}) | optimal LR: {eta:.4f} | Curvature(Rayleigh Quotient): {kappa:.4f} | Gradient: {g}")
  print('\n')
  optimizer.verify_curvature(x)
  print('\n')

At location: (0.52, 0.10) | optimal LR: 0.0361 | Curvature(Rayleigh Quotient): 19.5968 | Gradient: [1.13080925 7.4707704 ]


True κ:  16.329634
FD κ:    16.329634
Error:   3.02e-12


At location: (0.50, 0.05) | optimal LR: 0.0263 | Curvature(Rayleigh Quotient): 17.8773 | Gradient: [1.08780697 4.62979465]


True κ:  10.879420
FD κ:    10.879420
Error:   2.37e-12


At location: (0.47, 0.03) | optimal LR: 0.0225 | Curvature(Rayleigh Quotient): 15.2950 | Gradient: [1.05312634 3.28322668]


True κ:  6.403175
FD κ:    6.403175
Error:   1.18e-12


At location: (0.45, 0.02) | optimal LR: 0.0211 | Curvature(Rayleigh Quotient): 12.7094 | Gradient: [1.0229017  2.48558886]


True κ:  3.900204
FD κ:    3.900204
Error:   1.78e-12


At location: (0.44, 0.01) | optimal LR: 0.0207 | Curvature(Rayleigh Quotient): 10.5583 | Gradient: [0.99512455 1.95489367]


True κ:  2.759766
FD κ:    2.759766
Error:   1.97e-12


At location: (0.42, 0.01) | optimal LR: 0.0207 | Curvature(Rayleigh Quotient): 8.8939 | Gra

In [ ]:
class Rosenbrock:
    def function(self, x):
        return (1 - x[0])**2 + 100*(x[1] - x[0]**2)**2

    def gradients(self, x):
        dw1 = -2*(1 - x[0]) - 400*x[0]*(x[1] - x[0]**2)
        dw2 = 200*(x[1] - x[0]**2)
        return np.array([dw1, dw2])

In [ ]:
optimizer = CurvatureAndGradientBasedOptimizer(
    Rosenbrock(),
    base_learning_rate    = 0.001,
    minimum_learning_rate = 1e-6,
    maximum_learning_rate = 0.01,
    alpha = 0.9,
    beta  = 0.95
)

x = np.array([-1.0, 1.0])   # standard starting point, far from minimum

for i in range(1000):
    x, eta, kappa, g = optimizer.step(x)
    if i % 100 == 0:
        print(f"step {i:} | L={optimizer.fn.function(x):.6f} | x=({x[0]:.4f}, {x[1]:.4f}) | η={eta:.6f} | κ̄={kappa:.3f}")
        optimizer.verify_curvature(x)
        print('\n')

step    0 | L=3.999841 | x=(-1.0000, 1.0000) | η=0.000010 | κ̄=802.480
True κ:  798.678998
FD κ:    799.154239
Error:   4.75e-01


step  100 | L=3.993865 | x=(-0.9982, 0.9998) | η=0.000005 | κ̄=605.981
True κ:  562.125220
FD κ:    562.379837
Error:   2.55e-01


step  200 | L=3.990900 | x=(-0.9970, 0.9993) | η=0.000006 | κ̄=366.468
True κ:  315.000496
FD κ:    315.124970
Error:   1.24e-01


step  300 | L=3.987231 | x=(-0.9956, 0.9982) | η=0.000017 | κ̄=110.804
True κ:  64.400443
FD κ:    64.428170
Error:   2.77e-02


step  400 | L=3.914143 | x=(-0.9768, 0.9621) | η=0.001094 | κ̄=1.428
True κ:  -0.250738
FD κ:    -0.250720
Error:   1.74e-05


step  500 | L=2.942026 | x=(-0.7133, 0.5169) | η=0.003633 | κ̄=-0.348
True κ:  -0.401625
FD κ:    -0.401526
Error:   9.93e-05


step  600 | L=1.175060 | x=(-0.0838, 0.0090) | η=0.002351 | κ̄=0.417
True κ:  1.237650
FD κ:    1.235675
Error:   1.98e-03


step  700 | L=0.772405 | x=(0.1213, 0.0128) | η=0.000663 | κ̄=2.548
True κ:  2.655721
FD κ:    2.6